# India Mining Minerals: Data Pipeline and ML Model

This notebook follows the current project implementation: cleaning `combined_dataset.csv`, creating the synthetic training data, training the API model, evaluating it, reviewing its charts, and testing coordinate-based feature hydration.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None

from src.data_pipeline import MODEL_FEATURES, TARGET, prepare_data
pd.set_option('display.max_columns', 30)
print(f'Project root: {ROOT}')

## 1. Build and inspect the current datasets

`prepare_data()` is the same entry point used by `src.train`: it filters aggregate rows, assigns years, removes duplicate deposit-year records, imputes model features, and creates reproducible synthetic training rows.

In [ ]:
cleaned, training = prepare_data(samples=360, seed=42)
raw = pd.read_csv(ROOT / 'combined_dataset.csv')
print(f'Raw source rows: {len(raw):,}')
print(f'Cleaned inventory rows: {len(cleaned):,}')
print(f'Original training rows: {int((training["Synthetic"] == False).sum()):,}')
print(f'Synthetic/augmented training rows: {int(training["Synthetic"].sum()):,}')
print(f'Total ML training rows: {len(training):,}')
print(f'Financial years retained after cleaning: {sorted(cleaned["Year"].unique())}')
display(cleaned.head())

summary = pd.DataFrame({
    'raw_source_rows': [len(raw)],
    'inventory_rows': [len(cleaned)],
    'synthetic_training_rows': [int(training['Synthetic'].sum())],
    'states': [cleaned['State'].nunique()],
    'districts': [cleaned['District'].nunique()],
    'total_annual_production_tonnes': [cleaned[TARGET].sum()],
    'average_grade_pct': [pd.to_numeric(cleaned.get('Grade_pct'), errors='coerce').mean()],
    'missing_model_values': [int(cleaned[MODEL_FEATURES].isna().sum().sum())],
})
display(summary)
display(cleaned[MODEL_FEATURES + [TARGET]].describe(include='all').T)

## 2. Production trend

This aggregation matches the API's `/api/production/trend` endpoint.

In [ ]:
trend = cleaned.groupby('Year', as_index=False)[TARGET].sum().sort_values('Year')
display(trend)
plt.figure(figsize=(10, 5))
plt.bar(trend['Year'], trend[TARGET], color='steelblue')
plt.title('Annual Manganese Production by Reporting Year')
plt.xlabel('Year')
plt.ylabel('Production (tonnes)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 3. Train and evaluate the API model

The project training entry point uses median-imputed numeric features, most-frequent-imputed categorical features, one-hot encoding with unknown-category handling, and a 700-tree `RandomForestRegressor`. Validation groups real and synthetic siblings by Deposit_ID so augmented copies cannot cross the evaluation boundary.

In [ ]:
from src.train import train_model
metrics = train_model()
display(pd.DataFrame([{
    'model': metrics['model'],
    'target': metrics['target'],
    'R2': metrics['r2'],
    'MAE (tonnes)': metrics['mae'],
    'RMSE (tonnes)': metrics['rmse'],
    'training_rows': metrics['training_rows'],
    'cleaned_rows': metrics['cleaned_rows'],
}]))
print('Features:', ', '.join(metrics['features']))

In [ ]:
from IPython.display import Image, display
plots_dir = ROOT / 'plots'
display(Image(filename=str(plots_dir / 'ml_actual_vs_predicted.png')))
display(Image(filename=str(plots_dir / 'ml_feature_importance.png')))

## 4. Test coordinate-based feature hydration

The API uses `FeatureHydrator` to find the nearest cleaned district and nearest port when a prediction request supplies coordinates.

In [ ]:
from src.hydration import FeatureHydrator
hydrator = FeatureHydrator()
sample = cleaned.dropna(subset=['Latitude', 'Longitude']).iloc[0]
hydrated = hydrator.hydrate(float(sample['Latitude']), float(sample['Longitude']))
display(pd.Series({
    'input_latitude': hydrated['Latitude'],
    'input_longitude': hydrated['Longitude'],
    'hydrated_from_district': hydrated['hydrated_from_district'],
    'nearest_port': hydrated['nearest_port'],
    'distance_to_port_km': hydrated['Distance_to_Port_km'],
    'road_accessibility': hydrated.get('Road_Accessibility'),
    'soil_type': hydrated.get('Soil_Type'),
}).to_frame('value'))

## Outputs

- `data/cleaned_districts.csv` and `data/training_dataset.csv`
- `models/model.pkl` and `models/metrics.json`
- `plots/ml_actual_vs_predicted.png` and `plots/ml_feature_importance.png`

These are the same artifacts consumed by `main.py` and the frontend API.